# Scenario: Fixing the Fragmented Diagnoses Skew

In [3]:
import pandas as pd
import sqlite3
# creating dataset with fragmented casing for the same underlying medical condition
diagnostic_records = {    
    "record_id": [901, 902, 903, 904, 905, 906],
    "patient_id": ["P-201", "P-202", "P-203", "P-204", "P-205", "P-206"],
    "icd10_description": ["Diabetes", "diabetes", "Hypertension", "DIABETES", "hypertension", "Diabetes Mellitus"]
}
# adding dataset to dataframe
df_diagnostic_records = pd.DataFrame(diagnostic_records)
# creating sql and save dataframe to temp memory
connt = sqlite3.connect(":memory:")
df_diagnostic_records.to_sql("diagnoses_log", connt, index = False, if_exists = "replace")
# function to run query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("******************************** Text Normalization Audit Database is ready! ***************")

******************************** Text Normalization Audit Database is ready! ***************


# The Capitalization Fragmentation Scan

In [6]:
# SQL query for all data to review
all_data = "SELECT * FROM diagnoses_log"
print("********************* all data to review ***************")
display(run_query(all_data))
print()
occurrence_count = """
SELECT
    icd10_description,
    COUNT(*) AS occurrence_count
FROM diagnoses_log
GROUP BY icd10_description
ORDER BY occurrence_count DESC
"""
print("**************************** occurrence_count ***************")
display(run_query(occurrence_count))

********************* all data to review ***************


,record_id,patient_id,icd10_description
0,901,P-201,Diabetes
1,902,P-202,diabetes
2,903,P-203,Hypertension
3,904,P-204,DIABETES
4,905,P-205,hypertension
5,906,P-206,Diabetes Mellitus



**************************** occurrence_count ***************


,icd10_description,occurrence_count
0,hypertension,1
1,diabetes,1
2,Hypertension,1
3,Diabetes Mellitus,1
4,Diabetes,1
5,DIABETES,1


# Enforcing a Uniform Standard (Normalization)

In [10]:
""" SQL query that uses the LOWER() function to transform all entries to lowercase, and then group by that lowercase value to get a true, 
accurate count of patients with these conditions."""
uniform_standard = """
SELECT LOWER(icd10_description) AS normalized_diagnosis,
COUNT(*) AS true_patient_count
FROM diagnoses_log
GROUP BY normalized_diagnosis
"""
print("*************************** Uniform Standard ************")
display(run_query(uniform_standard))

*************************** Uniform Standard ************


,normalized_diagnosis,true_patient_count
0,diabetes,3
1,diabetes mellitus,1
2,hypertension,2
